# Vendor Products Data Analysis

This notebook loads product data from two sheets (Items and Collections), cleans the data, enriches the Collections sheet with missing columns from Items, and stacks both into one unified output file.

## Setup

Import pandas — the only library needed.

In [1]:
import pandas as pd

## Load Data

Read both sheets from the Excel file.

In [2]:
items = pd.read_excel('VendorProducts.xlsx', sheet_name='Items')
cols = pd.read_excel('VendorProducts.xlsx', sheet_name='Collections')

## Step 1: Clean Prices

Prices are stored as text with a currency suffix (`د.ع`) and thousand-separator commas. Remove both and convert to a clean integer.

In [3]:
items['Price'] = items['Price'].str.replace('د.ع', '').str.replace(',', '').astype(int)
cols['Price'] = cols['Price'].str.replace('د.ع', '').str.replace(',', '').astype(int)

## Step 2: Transform Collections Sheet

### 2a. Merge Name + Color

Append the Color (Arabic) to Item Name and Secondary Color (English) to Secondary Item Name. This gives each variant a unique product name that includes its color/size.

`fillna('')` handles products that have no color value. `str.strip()` removes extra spaces.

In [4]:
cols['Item Name'] = cols['Item Name'] + ' ' + cols['Color'].fillna('')
cols['Secondary Item Name'] = cols['Secondary Item Name'] + ' ' + cols['Secondary Color'].fillna('')
cols['Item Name'] = cols['Item Name'].str.strip()
cols['Secondary Item Name'] = cols['Secondary Item Name'].str.strip()

### 2b. Drop Unwanted Columns

Drop columns that are not in the final output: Color, Secondary Color, Size, Code, and Color Code.

In [5]:
cols.drop(columns=['Color', 'Secondary Color', 'Size', 'Code', 'Color Code'], inplace=True)

### 2c. Rename Columns to Match Items Sheet

Rename so column names are identical between both sheets (needed for concat later).

In [6]:
cols.rename(columns={
    'Item Id': 'Id',
    'Item Name': 'Name',
    'Secondary Item Name': 'Secondary Name'
}, inplace=True)

After renaming, Collections has these columns:
- `Id`, `Name`, `Secondary Name`, `Price`, `Barcode`, `Unit Level`, `Is Active`, `Picture`

## Step 3: Clean Items Sheet

Drop columns we don't need: Code, Sub Description, and Secondary Sub Description. Price was already cleaned in Step 1.

In [7]:
items.drop(columns=['Code', 'Sub Description', 'Secondary Sub Description'], inplace=True)

After dropping, Items has these 12 columns:
- `Picture`, `Id`, `Name`, `Secondary Name`, `Barcode`, `Price`, `Description`, `Secondary Description`, `Menu`, `Brand`, `Secondary Brand`, `Has Collections`

## Step 4: Fill Missing Columns in Collections

Collections is missing these columns from Items:
- `Description`, `Secondary Description`, `Menu`, `Brand`, `Secondary Brand`, `Has Collections`

We map them from the Items sheet by matching on `Id` (every product in Collections also exists in Items).

In [8]:
lookup = items[['Id', 'Description', 'Secondary Description', 'Menu', 'Brand', 'Secondary Brand', 'Has Collections']]
cols = pd.merge(cols, lookup, on='Id', how='left')

## Step 5: Drop Collection-Only Columns

Drop `Unit Level` and `Is Active` — these only exist in Collections and are not in the final Items column structure. Both sheets need the same columns for stacking.

In [9]:
cols.drop(columns=['Unit Level', 'Is Active'], inplace=True)

Now both sheets have the same 12 columns.

**Items:** `Picture`, `Id`, `Name`, `Secondary Name`, `Barcode`, `Price`, `Description`, `Secondary Description`, `Menu`, `Brand`, `Secondary Brand`, `Has Collections`

**Collections:** `Id`, `Name`, `Secondary Name`, `Price`, `Barcode`, `Picture`, `Description`, `Secondary Description`, `Menu`, `Brand`, `Secondary Brand`, `Has Collections`

## Step 6: Stack Both Sheets

Concatenate Items (24,702 rows) and Collections (14,486 rows) into one unified dataframe. `ignore_index=True` resets the row numbers.

In [10]:
final = pd.concat([items, cols], ignore_index=True)

## Step 7: Reorder Columns

Arrange columns to match the original Items sheet order.

In [11]:
final = final[['Picture', 'Id', 'Name', 'Secondary Name', 'Barcode', 'Price',
               'Description', 'Secondary Description', 'Menu', 'Brand',
               'Secondary Brand', 'Has Collections']]

## Step 8: Verify and Export

Check the final shape and a preview before saving.

In [14]:
print('Final shape:', final.shape)
final.tail(10)

Final shape: (39188, 12)


,Picture,Id,Name,Secondary Name,Barcode,Price,Description,Secondary Description,Menu,Brand,Secondary Brand,Has Collections
39178,https://www.storeakmedia.com/storeak-erp/Store...,332027,كونسيلر فوري مضاد للشيخوخة 6 مل منير 05,Instant Anti Age Eraser Concealer 6 ml 05 Brig...,3600531396831,17250,كونسيلر مصحح بخصائص مضادة للشيخوخة تخفي علامات...,A corrective concealer with anti aging propert...,"المكياج‎ -> الوجه‎ -> كونسيلر & مصحح,المكياج‎ ...",ميبيلين,Maybelline,Yes
39179,https://www.storeakmedia.com/storeak-erp/Store...,332027,كونسيلر فوري مضاد للشيخوخة 6 مل فاتح 01,Instant Anti Age Eraser Concealer 6 ml 01 Light,3600530733842,17250,كونسيلر مصحح بخصائص مضادة للشيخوخة تخفي علامات...,A corrective concealer with anti aging propert...,"المكياج‎ -> الوجه‎ -> كونسيلر & مصحح,المكياج‎ ...",ميبيلين,Maybelline,Yes
39180,https://www.storeakmedia.com/storeak-erp/Store...,332027,كونسيلر فوري مضاد للشيخوخة 6 مل نيود 02,Instant Anti Age Eraser Concealer 6 ml 02 Nude,3600530733859,17250,كونسيلر مصحح بخصائص مضادة للشيخوخة تخفي علامات...,A corrective concealer with anti aging propert...,"المكياج‎ -> الوجه‎ -> كونسيلر & مصحح,المكياج‎ ...",ميبيلين,Maybelline,Yes
39181,https://www.storeakmedia.com/storeak-erp/Store...,332027,كونسيلر فوري مضاد للشيخوخة 6 مل رملي 07,Instant Anti Age Eraser Concealer 6 ml 07 Sand,3600531465247,17250,كونسيلر مصحح بخصائص مضادة للشيخوخة تخفي علامات...,A corrective concealer with anti aging propert...,"المكياج‎ -> الوجه‎ -> كونسيلر & مصحح,المكياج‎ ...",ميبيلين,Maybelline,Yes
39182,https://www.storeakmedia.com/storeak-erp/Store...,332027,كونسيلر فوري مضاد للشيخوخة 6 مل 06 محايد,Instant Anti Age Eraser Concealer 6 ml 06 Neutral,3600531396855,17250,كونسيلر مصحح بخصائص مضادة للشيخوخة تخفي علامات...,A corrective concealer with anti aging propert...,"المكياج‎ -> الوجه‎ -> كونسيلر & مصحح,المكياج‎ ...",ميبيلين,Maybelline,Yes
39183,https://www.storeakmedia.com/storeak-erp/Store...,332027,كونسيلر فوري مضاد للشيخوخة 6 مل 03 معتدل,Instant Anti Age Eraser Concealer 6 ml 03 Fair,3600530733866,17250,كونسيلر مصحح بخصائص مضادة للشيخوخة تخفي علامات...,A corrective concealer with anti aging propert...,"المكياج‎ -> الوجه‎ -> كونسيلر & مصحح,المكياج‎ ...",ميبيلين,Maybelline,Yes
39184,https://www.storeakmedia.com/storeak-erp/Store...,316612,غسول الفم للثة الصحية بالنعناع النقي 500 مل,Healthy Gums Mouthwash Pure Mint 500 ml,697029058770,17000,غسول الفم والاسنان لتقليل نزيف والتهابات اللثة...,Mouthwash and teeth to reduce bleeding and inf...,العناية بالبشرة‎ -> العناية بالفم‎ -> غرغرة & ...,ثيرابريث,TheraBreath,Yes
39185,https://www.storeakmedia.com/storeak-erp/Store...,316612,غسول الفم للثة الصحية بالنعناع النقي 88.7 مل,Healthy Gums Mouthwash Pure Mint 88.7 ml,697029602478,6500,غسول الفم والاسنان لتقليل نزيف والتهابات اللثة...,Mouthwash and teeth to reduce bleeding and inf...,العناية بالبشرة‎ -> العناية بالفم‎ -> غرغرة & ...,ثيرابريث,TheraBreath,Yes
39186,NaN,316610,غسول فم منعش بالنعناع الخفيف 500 مل,Fresh Breath Oral Rinse Mild Mint 500 ml,697029041260,17000,غسول الفم والاسنان يعمل على تحييد البكتيريا ال...,Mouthwash neutralizes sulfur producing bacteri...,العناية بالبشرة‎ -> العناية بالفم‎ -> غرغرة & ...,ذا بريث كو,The Breath.Co,Yes
39187,https://www.storeakmedia.com/storeak-erp/Store...,316610,غسول فم منعش بالنعناع الخفيف 88.7 مل,Fresh Breath Oral Rinse Mild Mint 88.7 ml,697029532409,6500,غسول الفم والاسنان يعمل على تحييد البكتيريا ال...,Mouthwash neutralizes sulfur producing bacteri...,العناية بالبشرة‎ -> العناية بالفم‎ -> غرغرة & ...,ذا بريث كو,The Breath.Co,Yes


In [16]:
final.to_excel('AnalysedVendorProducts.xlsx', index=False)

Done. The output file `VendorProducts_Final.xlsx` has all Items rows followed by all Collections rows (stacked vertically) with matching 12-column structure.